# R3m 原始工件恢复（CPU only）
在原有 bingjiangli/minicells 中导入，关闭 GPU，保留现有 HF_TOKEN，然后 Run All。仅恢复固定版本的原始 Tokenizer、Manifest、窗口元数据；不训练、不评价、不上传、不更改注册。完成后 Save Version 并回传 RECOVERY_PROOF.json。

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import uuid

SOURCE_SHA = "71b8cdb7a870309cdb9769ea80f0f1a810dee6be"
ROOT = Path('/kaggle/working/r3m-recovery-code-' + SOURCE_SHA[:12])
OUTPUT = Path('/kaggle/working/r3m-artifact-recovery-' + uuid.uuid4().hex[:12])

def execute(*args):
    return subprocess.run(list(map(str, args)), check=True)

if importlib.util.find_spec('huggingface_hub') is None:
    execute(sys.executable, '-m', 'pip', 'install', '--quiet', 'huggingface_hub>=0.24,<2')
if not os.environ.get('HF_TOKEN', '').strip():
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN').strip()
    except Exception:
        raise RuntimeError('Enable the existing HF_TOKEN secret in the original bingjiangli/minicells Notebook. No new token is requested.') from None
if not os.environ.get('HF_TOKEN', '').strip():
    raise RuntimeError('Existing HF_TOKEN is empty')
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS='1', HF_HUB_DISABLE_TELEMETRY='1')
if not (ROOT / '.git').exists():
    execute('git', 'init', '--quiet', ROOT)
    execute('git', '-C', ROOT, 'remote', 'add', 'origin', 'https://github.com/ArcheLabs/mini-cells.git')
    execute('git', '-C', ROOT, 'config', 'remote.origin.promisor', 'true')
    execute('git', '-C', ROOT, 'config', 'remote.origin.partialclonefilter', 'blob:none')
    execute('git', '-C', ROOT, 'sparse-checkout', 'init', '--no-cone')
    execute('git', '-C', ROOT, 'sparse-checkout', 'set', '--no-cone',
            '/shared/python/minicells_artifacts/', '/shared/python/minicells_runtime/',
            '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/recover_original_artifacts.py',
            '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/config/frozen.json')
    execute('git', '-C', ROOT, 'fetch', '--depth=1', '--filter=blob:none', 'origin', SOURCE_SHA)
    execute('git', '-C', ROOT, 'checkout', '--detach', 'FETCH_HEAD')
actual = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if actual != SOURCE_SHA:
    raise RuntimeError('Recovery source identity mismatch')
print({'source_sha': SOURCE_SHA, 'scope': 'CPU_METADATA_RECOVERY_ONLY', 'gpu_training': False}, flush=True)
try:
    execute(sys.executable, ROOT / 'research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/recover_original_artifacts.py', '--output', OUTPUT)
finally:
    print('Save Version retains recovery evidence: ' + str(OUTPUT), flush=True)
print('Original bytes recovered. Registration and Window Gate remain BLOCKED pending independent review.')
